# Phase 2 Evaluation Notebook
Intrinsic and extrinsic evaluation of Hindi and Nepali checkpoints.


## 1. Configuration & Paths


In [ ]:

from pathlib import Path

# Data Roots
KAGGLE_DATASET_ROOT = Path("/kaggle/input/datasets/negative00/phase2-cloud-data")

# Modify these if your checkpoints are attached in a different dataset
HINDI_CKPT = Path("/kaggle/input/hindi-checkpoint/step_20000.pt")
NEPALI_CKPT = Path("/kaggle/input/nepali-checkpoint/step_20000.pt")

HINDI_TOKENIZER = KAGGLE_DATASET_ROOT / "hindi/tokenizer/models/hindi_bpe_10k_final.model"
NEPALI_TOKENIZER = KAGGLE_DATASET_ROOT / "nepali/tokenizer/models/nepali_bpe_10k_assignment.model"

HINDI_VAL_DIR = KAGGLE_DATASET_ROOT / "hindi/data/splits/val"
HINDI_TEST_DIR = KAGGLE_DATASET_ROOT / "hindi/data/splits/test"

NEPALI_VAL_DIR = KAGGLE_DATASET_ROOT / "nepali/data/final/val"
NEPALI_TEST_DIR = KAGGLE_DATASET_ROOT / "nepali/data/final/test"

OUTPUT_DIR = Path("/kaggle/working/phase2_evaluation")

# Evaluation Configuration
MAX_SEQ_LEN = 512
EVAL_DOCUMENTS_LIMIT = 500  # Number of documents to evaluate to save time, None for full
GENERATION_PROMPTS = 5
PROMPT_LENGTH = 10
MAX_GEN_TOKENS = 50
TEMPERATURES = [None, 0.5, 1.0, 1.5] # None means greedy


## 2. Imports & Environment


In [ ]:

import os
import json
import math
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import sentencepiece as spm
from pathlib import Path
from tqdm.auto import tqdm
import seaborn as sns
from collections import Counter

try:
    import sacrebleu
    from rouge_score import rouge_scorer
    METRICS_AVAILABLE = True
except ImportError:
    print("Warning: sacrebleu or rouge_score not installed. Generation metrics will be skipped.")
    print("Run `!pip install sacrebleu rouge-score` to enable them.")
    METRICS_AVAILABLE = False


## 3. Source Implementations (Config, Model, Data)


In [ ]:
from dataclasses import dataclass

@dataclass
class LMConfig:
    vocab_size: int = 10000
    d_model: int = 448
    n_layers: int = 8
    n_heads: int = 7
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    bias: bool = True


In [ ]:
import math
import torch
import torch.nn as nn

class InputEmbeddings(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.max_seq_len = config.max_seq_len
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"Sequence length {T} exceeds maximum sequence length {self.max_seq_len}")
            
        device = input_ids.device
        position_ids = torch.arange(0, T, dtype=torch.long, device=device)
        
        token_embeddings = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(position_ids)
        
        x = token_embeddings + position_embeddings
        x = self.dropout(x)
        return x

class CausalSelfAttention(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        if config.d_model % config.n_heads != 0:
            raise ValueError(f"d_model ({config.d_model}) must be divisible by n_heads ({config.n_heads})")
            
        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.max_seq_len = config.max_seq_len
        
        self.q_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        
        # Causal mask register_buffer ensures it moves to correct device
        mask = torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)).view(
            1, 1, config.max_seq_len, config.max_seq_len
        )
        self.register_buffer("bias_mask", mask)
        
    def forward(self, x, return_attention=False):
        B, T, C = x.size() # B, T, d_model
        
        if T > self.max_seq_len:
            raise ValueError(f"Sequence length {T} exceeds maximum sequence length {self.max_seq_len}")
            
        # Linear projections
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        
        # Reshape to [B, T, n_heads, head_dim] -> [B, n_heads, T, head_dim]
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Scaled dot-product attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        
        # Causal mask applied BEFORE softmax
        att = att.masked_fill(self.bias_mask[:, :, :T, :T] == 0, float('-inf'))
        
        # Softmax
        att = torch.nn.functional.softmax(att, dim=-1)
        
        # Attention probabilities dropout
        att_dropped = self.attn_dropout(att)
        
        # Multiply with values
        y = att_dropped @ v
        
        # Re-assemble all head outputs side by side
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # Output projection
        y = self.resid_dropout(self.out_proj(y))
        
        if return_attention:
            return y, att
        return y


class FeedForward(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff, bias=config.bias),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_ff, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.ffn = FeedForward(config)

    def forward(self, x, return_attention=False):
        # Pre-LayerNorm: x = x + Attention(LayerNorm(x))
        if return_attention:
            attn_out, att_weights = self.attn(self.ln_1(x), return_attention=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.ln_1(x))

        # Pre-LayerNorm: x = x + FFN(LayerNorm(x))
        x = x + self.ffn(self.ln_2(x))

        if return_attention:
            return x, att_weights
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, config: LMConfig):
        super().__init__()
        self.config = config

        # Input representation
        self.embedding = InputEmbeddings(config)

        # Transformer blocks
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])

        # Final LayerNorm
        self.ln_f = nn.LayerNorm(config.d_model)

        # LM head: weight is tied to the token embedding, but has its own bias
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=config.bias)

        # Weight tying: LM head weight IS the token embedding weight
        self.lm_head.weight = self.embedding.token_embedding.weight

    def forward(self, input_ids, return_attention=False):
        # input_ids: [B, T]
        x = self.embedding(input_ids)

        attentions = [] if return_attention else None

        for block in self.blocks:
            if return_attention:
                x, att = block(x, return_attention=True)
                attentions.append(att)
            else:
                x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)  # [B, T, vocab_size]

        if return_attention:
            return logits, attentions
        return logits


In [ ]:
"""
Phase 2 — Stage 5: Data Preparation Pipeline

Converts JSONL corpora into memory-mapped uint16 .bin files for
efficient language model training.

Supports both Hindi and Nepali through configuration arguments.
All paths are supplied externally for portability across
local / Kaggle / Colab environments.
"""

import argparse
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import sentencepiece as spm


# ---------------------------------------------------------------------------
# Tokenization / conversion
# ---------------------------------------------------------------------------

def stream_jsonl_texts(input_dir: Path, text_field: str = "text"):
    """Yield text strings from all .jsonl files in input_dir (sorted for determinism)."""
    jsonl_files = sorted(input_dir.glob("*.jsonl"))
    if not jsonl_files:
        raise FileNotFoundError(f"No .jsonl files found in {input_dir}")
    for jsonl_path in jsonl_files:
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                doc = json.loads(line)
                text = doc.get(text_field)
                if text and isinstance(text, str) and text.strip():
                    yield text.strip()
                else:
                    yield None  # signal a skip


def convert_split(
    input_dir: Path,
    output_path: Path,
    tokenizer_path: Path,
    text_field: str = "text",
):
    """
    Stream JSONL documents, tokenize with SentencePiece, and write
    token IDs as a flat uint16 binary file.

    Returns a dict of statistics.
    """
    # Load tokenizer
    sp = spm.SentencePieceProcessor()
    sp.Load(str(tokenizer_path))
    vocab_size = sp.GetPieceSize()

    if vocab_size > 65535:
        raise ValueError(
            f"Vocabulary size {vocab_size} exceeds uint16 max (65535)"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    n_docs = 0
    n_skipped = 0
    n_tokens = 0
    token_min = 65535
    token_max = 0

    t0 = time.time()

    with open(output_path, "wb") as out_f:
        for text in stream_jsonl_texts(input_dir, text_field):
            if text is None:
                n_skipped += 1
                n_docs += 1
                continue

            ids = sp.Encode(text, out_type=int)
            if not ids:
                n_skipped += 1
                n_docs += 1
                continue

            arr = np.array(ids, dtype=np.uint16)
            arr.tofile(out_f)

            n_tokens += len(ids)
            token_min = min(token_min, int(arr.min()))
            token_max = max(token_max, int(arr.max()))
            n_docs += 1

    elapsed = time.time() - t0
    file_bytes = output_path.stat().st_size if output_path.exists() else 0

    stats = {
        "input_dir": input_dir.name,
        "output_path": output_path.name,
        "tokenizer": tokenizer_path.name,
        "vocab_size": vocab_size,
        "dtype": "uint16",
        "documents_processed": n_docs,
        "documents_skipped": n_skipped,
        "token_count": n_tokens,
        "token_min": token_min if n_tokens > 0 else None,
        "token_max": token_max if n_tokens > 0 else None,
        "bytes_written": file_bytes,
        "elapsed_seconds": round(elapsed, 3),
        "tokens_per_second": round(n_tokens / elapsed, 1) if elapsed > 0 else 0,
    }

    return stats


def save_metadata(output_dir: Path, language: str, split_stats: dict,
                  tokenizer_name: str):
    """Save metadata.json alongside the .bin files."""
    meta = {
        "language": language,
        "tokenizer_model": tokenizer_name,
        "vocab_size": 10000,
        "dtype": "uint16",
        "context_length": 512,
        "conversion_timestamp": datetime.now(timezone.utc).isoformat(),
        "splits": split_stats,
    }
    meta_path = output_dir / "metadata.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    return meta_path


# ---------------------------------------------------------------------------
# Dataset reader (memory-mapped)
# ---------------------------------------------------------------------------

class TokenDataset:
    """
    Memory-mapped dataset over a flat uint16 .bin file.

    Each sample returns:
        x: tokens[i : i + context_length]       (input)
        y: tokens[i+1 : i + context_length + 1]  (target)
    """

    def __init__(self, bin_path: Path, context_length: int = 512):
        self.bin_path = Path(bin_path)
        if not self.bin_path.exists():
            raise FileNotFoundError(f"Binary file not found: {self.bin_path}")

        self.data = np.memmap(self.bin_path, dtype=np.uint16, mode="r")
        self.context_length = context_length
        self.n_tokens = len(self.data)

        # We need at least context_length + 1 tokens for one sample
        if self.n_tokens < context_length + 1:
            raise ValueError(
                f"Token file has {self.n_tokens} tokens but needs at least "
                f"{context_length + 1} for context_length={context_length}"
            )

    def __len__(self):
        """Number of valid starting positions."""
        return self.n_tokens - self.context_length

    def __getitem__(self, idx):
        """Return (x, y) token arrays at position idx."""
        if idx < 0 or idx >= len(self):
            raise IndexError(
                f"Index {idx} out of range for dataset with {len(self)} positions"
            )
        chunk = self.data[idx : idx + self.context_length + 1].astype(np.int64)
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

    def random_sample(self, rng=None):
        """Return a random (x, y) pair."""
        if rng is None:
            rng = np.random.default_rng()
        idx = rng.integers(0, len(self))
        return self[idx]


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------




## 4. Output Setup & Device Validation


In [ ]:

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures" / "attention").mkdir(parents=True, exist_ok=True)

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Available: True")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    t = torch.cuda.get_device_properties(0).total_memory
    print(f"Total VRAM: {t / 1024**3:.2f} GB")
    device = torch.device("cuda")
else:
    print("GPU Available: False")
    device = torch.device("cpu")
    print("Warning: CUDA is unavailable. Evaluation will run on CPU.")


## 5. Checkpoint Loader


In [ ]:
def load_eval_model(ckpt_path):
    print(f"Loading checkpoint from {ckpt_path}...")

    state = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False,
    )

    if not isinstance(state, dict):
        raise TypeError("Checkpoint must be a dictionary.")

    if "model_state" not in state:
        raise KeyError("Checkpoint does not contain 'model_state'.")

    checkpoint_cfg = state.get("model_cfg")

    if checkpoint_cfg is not None:
        if isinstance(checkpoint_cfg, dict):
            cfg = LMConfig(**checkpoint_cfg)
        else:
            cfg = checkpoint_cfg
    else:
        cfg = LMConfig(
            vocab_size=10000,
            d_model=448,
            n_layers=8,
            n_heads=7,
            d_ff=2048,
            max_seq_len=512,
            dropout=0.1,
            bias=True,
        )

    model = DecoderOnlyLM(cfg)

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    if parameter_count != 25_871_504:
        raise AssertionError(
            f"Unexpected parameter count: {parameter_count}"
        )

    incompatible = model.load_state_dict(
        state["model_state"],
        strict=True,
    )

    missing_keys = list(incompatible.missing_keys)
    unexpected_keys = list(incompatible.unexpected_keys)

    if missing_keys:
        raise RuntimeError(
            f"Missing checkpoint keys: {missing_keys}"
        )

    if unexpected_keys:
        raise RuntimeError(
            f"Unexpected checkpoint keys: {unexpected_keys}"
        )

    step = state.get("step", None)

    print(f"Loaded checkpoint step: {step}")
    print(f"Parameter count: {parameter_count}")
    print(f"Missing keys: {missing_keys}")
    print(f"Unexpected keys: {unexpected_keys}")

    return model, step


## 6. Intrinsic Evaluation Logic


In [ ]:

@torch.inference_mode()
def evaluate_intrinsic(model, tokenizer_path, data_dir, device, limit=None):
    model = model.to(device)
    sp = spm.SentencePieceProcessor(model_file=str(tokenizer_path))
    
    total_loss = 0.0
    total_tokens = 0
    total_bytes = 0
    total_docs = 0
    
    criterion = nn.CrossEntropyLoss(reduction='sum')
    
    for doc in stream_jsonl_texts(Path(data_dir), "text"):
        if limit is not None and total_docs >= limit:
            break
            
        # NLL / BPB calculation uses UTF-8 bytes
        doc_bytes = len(doc.encode("utf-8"))
        
        ids = sp.encode(doc, out_type=int)
        if len(ids) < 2:
            continue
            
        # Chunk into context lengths
        for i in range(0, len(ids) - 1, MAX_SEQ_LEN):
            chunk = ids[i : i + MAX_SEQ_LEN + 1]
            if len(chunk) < 2:
                break
                
            x = torch.tensor(chunk[:-1], dtype=torch.long).unsqueeze(0).to(device)
            y = torch.tensor(chunk[1:], dtype=torch.long).unsqueeze(0).to(device)
            
            logits = model(x) # (1, seq, vocab)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            
            total_loss += loss.item()
            total_tokens += x.size(1)
            
        total_bytes += doc_bytes
        total_docs += 1

    avg_loss = total_loss / total_tokens if total_tokens > 0 else 0
    ppl = math.exp(avg_loss) if avg_loss < 20 else float('inf')
    
    # BPB calculation: total NLL (base e) converted to base 2, divided by total bytes
    # total_loss is sum of -log_e(p)
    bpb = (total_loss / math.log(2)) / total_bytes if total_bytes > 0 else 0
    
    return {
        "docs_evaluated": total_docs,
        "tokens_evaluated": total_tokens,
        "bytes_evaluated": total_bytes,
        "loss": avg_loss,
        "perplexity": ppl,
        "bpb": bpb
    }


## 7. Generation Logic


In [ ]:

@torch.inference_mode()
def generate_text(model, sp, prompt, max_tokens, temperature=None):
    model.eval()
    ids = sp.encode(prompt, out_type=int)
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    
    for _ in range(max_tokens):
        if x.size(1) > MAX_SEQ_LEN:
            x_cond = x[:, -MAX_SEQ_LEN:]
        else:
            x_cond = x
            
        logits = model(x_cond)
        next_logits = logits[0, -1, :]
        
        if temperature is None or temperature == 0.0:
            next_id = torch.argmax(next_logits).item()
        else:
            next_logits = next_logits / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)
        
    return sp.decode(x[0].tolist())

def extract_prompts(data_dir, tokenizer_path, num_prompts, prompt_length):
    sp = spm.SentencePieceProcessor(model_file=str(tokenizer_path))
    
    prompts = []
    references = []
    
    for doc in stream_jsonl_texts(Path(data_dir), "text"):
        if len(prompts) >= num_prompts:
            break
        ids = sp.encode(doc, out_type=int)
        if len(ids) > prompt_length + 20:
            prompt_ids = ids[:prompt_length]
            ref_ids = ids[prompt_length:prompt_length+MAX_GEN_TOKENS]
            prompts.append(sp.decode(prompt_ids))
            references.append(sp.decode(ref_ids))
            
    return prompts, references


## 8. Metric Definitions


In [ ]:

def calculate_distinct_n(text, n):
    tokens = text.split()
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return len(set(ngrams)) / len(ngrams)

def calculate_repetition_rate(text):
    tokens = text.split()
    if not tokens:
        return 0.0
    counts = Counter(tokens)
    # Proportion of tokens that appear more than once
    repeated = sum(count for word, count in counts.items() if count > 1)
    return repeated / len(tokens)

def compute_generation_metrics(generations, references):
    if not METRICS_AVAILABLE:
        return {"error": "metrics_libraries_unavailable"}
        
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    
    bleus = []
    chrfs = []
    rouge_ls = []
    dist1s = []
    dist2s = []
    reps = []
    
    for gen, ref in zip(generations, references):
        # sacrebleu expects list of references
        b = sacrebleu.sentence_bleu(gen, [ref]).score
        c = sacrebleu.sentence_chrf(gen, [ref]).score
        r = scorer.score(ref, gen)['rougeL'].fmeasure
        
        bleus.append(b)
        chrfs.append(c)
        rouge_ls.append(r)
        dist1s.append(calculate_distinct_n(gen, 1))
        dist2s.append(calculate_distinct_n(gen, 2))
        reps.append(calculate_repetition_rate(gen))
        
    return {
        "bleu": np.mean(bleus),
        "chrf": np.mean(chrfs),
        "rougeL": np.mean(rouge_ls),
        "distinct_1": np.mean(dist1s),
        "distinct_2": np.mean(dist2s),
        "repetition_rate": np.mean(reps)
    }


## 9. Attention Analysis Logic


In [ ]:

@torch.inference_mode()
def analyze_attention(model, sp, text, lang_prefix):
    model.eval()
    ids = sp.encode(text, out_type=int)
    # truncate for plot readability
    max_len = 32
    ids = ids[:max_len]
    tokens = [sp.decode([idx]) for idx in ids]
    
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    _, attentions = model(x, return_attention=True)
    
    # attentions is list of (batch, heads, seq, seq)
    layer_0 = attentions[0][0].cpu().numpy() # (heads, seq, seq)
    layer_7 = attentions[7][0].cpu().numpy()
    
    # Calculate Mean Attention Distance and Entropy
    seq_len = x.size(1)
    
    def calc_stats(attn_matrix):
        # attn_matrix: (heads, seq, seq)
        # Entropy = -sum(p * log p)
        epsilon = 1e-10
        entropy = -np.sum(attn_matrix * np.log(attn_matrix + epsilon), axis=-1) # (heads, seq)
        mean_entropy = np.mean(entropy)
        
        # Distance = sum(p * |i - j|)
        indices = np.arange(seq_len)
        dist_matrix = np.abs(indices.reshape(-1, 1) - indices.reshape(1, -1))
        distances = np.sum(attn_matrix * dist_matrix, axis=-1)
        mean_distance = np.mean(distances)
        
        return mean_entropy, mean_distance

    e0, d0 = calc_stats(layer_0)
    e7, d7 = calc_stats(layer_7)
    
    # Plotting Layer 0 Head 0, Layer 7 Head 0
    def plot_heatmap(matrix, layer_idx):
        plt.figure(figsize=(8, 6))
        sns.heatmap(matrix[0], xticklabels=tokens, yticklabels=tokens, cmap='viridis')
        plt.title(f"{lang_prefix.capitalize()} - Layer {layer_idx} Head 0")
        plt.xlabel("Key Token")
        plt.ylabel("Query Token")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "figures" / "attention" / f"{lang_prefix}_layer{layer_idx}.png")
        plt.close()
        
    plot_heatmap(layer_0, 0)
    plot_heatmap(layer_7, 7)
    
    return {
        "layer_0_mean_entropy": float(e0),
        "layer_0_mean_distance": float(d0),
        "layer_7_mean_entropy": float(e7),
        "layer_7_mean_distance": float(d7)
    }


## 10. Execution Pipeline


In [ ]:

summary_results = {}
intrinsic_results = {}
generation_results = {}
generation_samples = []
attention_results = {}

languages = [
    ("hindi", HINDI_CKPT, HINDI_TOKENIZER, HINDI_VAL_DIR, HINDI_TEST_DIR),
    ("nepali", NEPALI_CKPT, NEPALI_TOKENIZER, NEPALI_VAL_DIR, NEPALI_TEST_DIR)
]

for lang, ckpt, tok_path, val_dir, test_dir in languages:
    print(f"\n{'='*50}\nEvaluating {lang.upper()}\n{'='*50}")
    
    try:
        model, step = load_eval_model(ckpt)
        model = model.to(device)
    except FileNotFoundError as e:
        print(f"Skipping {lang} due to missing checkpoint: {e}")
        continue
        
    sp = spm.SentencePieceProcessor(model_file=str(tok_path))
    
    # 1. Intrinsic
    print("Running Intrinsic Evaluation on Validation...")
    val_int = evaluate_intrinsic(model, tok_path, val_dir, device, limit=EVAL_DOCUMENTS_LIMIT)
    print(f"Val Loss: {val_int['loss']:.4f}, PPL: {val_int['perplexity']:.4f}, BPB: {val_int['bpb']:.4f}")
    
    print("Running Intrinsic Evaluation on Test...")
    test_int = evaluate_intrinsic(model, tok_path, test_dir, device, limit=EVAL_DOCUMENTS_LIMIT)
    print(f"Test Loss: {test_int['loss']:.4f}, PPL: {test_int['perplexity']:.4f}, BPB: {test_int['bpb']:.4f}")
    
    intrinsic_results[lang] = {"val": val_int, "test": test_int}
    
    # 2. Generation & Metrics
    print("Running Generation...")
    prompts, references = extract_prompts(val_dir, tok_path, GENERATION_PROMPTS, PROMPT_LENGTH)
    lang_gens = []
    
    for t in TEMPERATURES:
        t_label = f"temp_{t}" if t else "greedy"
        gens = []
        for p in prompts:
            g = generate_text(model, sp, p, MAX_GEN_TOKENS, temperature=t)
            gens.append(g)
            
        metrics = compute_generation_metrics(gens, references)
        generation_results.setdefault(lang, {})[t_label] = metrics
        
        for p, r, g in zip(prompts, references, gens):
            lang_gens.append({
                "language": lang,
                "temperature": t_label,
                "prompt": p,
                "reference": r,
                "generated": g
            })
            
    generation_samples.extend(lang_gens)
    
    # 3. Attention
    print("Running Attention Analysis...")
    attn_metrics = analyze_attention(model, sp, prompts[0] + references[0], lang)
    attention_results[lang] = attn_metrics
    
    # 4. Summary buildup
    # Use greedy metrics for summary
    g_mets = generation_results[lang].get("greedy", {})
    summary_results[lang] = {
        "checkpoint_path": str(ckpt),
        "checkpoint_step": step,
        "parameter_count": sum(p.numel() for p in model.parameters()),
        "tokenizer_path": str(tok_path),
        "validation_loss": val_int["loss"],
        "test_loss": test_int["loss"],
        "validation_perplexity": val_int["perplexity"],
        "test_perplexity": test_int["perplexity"],
        "validation_bpb": val_int["bpb"],
        "test_bpb": test_int["bpb"],
        "bleu": g_mets.get("bleu"),
        "chrf": g_mets.get("chrf"),
        "rougeL": g_mets.get("rougeL"),
        "repetition_rate": g_mets.get("repetition_rate"),
        "distinct_1": g_mets.get("distinct_1"),
        "distinct_2": g_mets.get("distinct_2"),
        "attention_entropy": attn_metrics["layer_7_mean_entropy"],
        "mean_attention_distance": attn_metrics["layer_7_mean_distance"],
        "documents_evaluated": val_int["docs_evaluated"],
        "tokens_evaluated": val_int["tokens_evaluated"],
        "generation_settings": {"max_tokens": MAX_GEN_TOKENS, "prompt_length": PROMPT_LENGTH},
        "metrics_available": METRICS_AVAILABLE
    }

    del model
    torch.cuda.empty_cache()


## 11. Save Output Files


In [ ]:

# Intrinsic Metrics
with open(OUTPUT_DIR / "intrinsic_metrics.json", "w", encoding="utf-8") as f:
    json.dump(intrinsic_results, f, indent=2, ensure_ascii=False)

# Generation Metrics
with open(OUTPUT_DIR / "generation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(generation_results, f, indent=2, ensure_ascii=False)

# Attention Metrics
with open(OUTPUT_DIR / "attention_metrics.json", "w", encoding="utf-8") as f:
    json.dump(attention_results, f, indent=2, ensure_ascii=False)

# Generation Samples JSON
with open(OUTPUT_DIR / "generation_samples.json", "w", encoding="utf-8") as f:
    json.dump(generation_samples, f, indent=2, ensure_ascii=False)
    
# Evaluation Summary
with open(OUTPUT_DIR / "evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_results, f, indent=2, ensure_ascii=False)

# Generation Samples Markdown
with open(OUTPUT_DIR / "generation_samples.md", "w", encoding="utf-8") as f:
    f.write("# Generation Samples\n\n")
    for s in generation_samples:
        f.write(f"### {s['language'].capitalize()} - {s['temperature']}\n")
        f.write(f"**Prompt:** {s['prompt']}\n\n")
        f.write(f"**Reference:** {s['reference']}\n\n")
        f.write(f"**Generated:** {s['generated']}\n\n")
        f.write("---\n\n")

# Attention Summary Markdown
with open(OUTPUT_DIR / "attention_summary.md", "w", encoding="utf-8") as f:
    f.write("# Attention Analysis Summary\n\n")
    for lang, mets in attention_results.items():
        f.write(f"### {lang.capitalize()}\n")
        f.write(f"- Layer 0 Mean Entropy: {mets['layer_0_mean_entropy']:.4f}\n")
        f.write(f"- Layer 0 Mean Distance: {mets['layer_0_mean_distance']:.4f}\n")
        f.write(f"- Layer 7 Mean Entropy: {mets['layer_7_mean_entropy']:.4f}\n")
        f.write(f"- Layer 7 Mean Distance: {mets['layer_7_mean_distance']:.4f}\n\n")
        
print(f"Evaluation complete. All results saved to {OUTPUT_DIR}")


## 12. Complete Missing Generation Metrics (Post-Hoc)
Computes BLEU, chrF, ROUGE-L, repetition_rate, distinct_1, and distinct_2 from saved generation samples.


In [ ]:
import subprocess
import sys
import json
import csv
import numpy as np
from pathlib import Path
from collections import Counter

# Install metric libraries if not available
def _ensure_installed(pkg_import, pkg_pip):
    try:
        __import__(pkg_import)
    except ImportError:
        print(f"Installing {pkg_pip}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_pip])

_ensure_installed("sacrebleu", "sacrebleu")
_ensure_installed("rouge_score", "rouge-score")

import sacrebleu
from rouge_score import rouge_scorer

print(f"sacrebleu version: {sacrebleu.__version__}")
print("rouge_score imported successfully")

# ---------- helper functions ----------

def calc_distinct_n(text, n):
    tokens = text.split()
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return len(set(ngrams)) / len(ngrams) if ngrams else 0.0

def calc_repetition_rate(text):
    tokens = text.split()
    if not tokens:
        return 0.0
    counts = Counter(tokens)
    repeated = sum(count for count in counts.values() if count > 1)
    return repeated / len(tokens)

# ---------- main computation ----------

samples_path = OUTPUT_DIR / "generation_samples.json"
if not samples_path.exists():
    raise FileNotFoundError(f"generation_samples.json not found at {samples_path}")

with open(samples_path, 'r', encoding='utf-8') as f:
    samples = json.load(f)

print(f"Loaded {len(samples)} generation samples from {samples_path}")
print(f"Sample keys: {list(samples[0].keys())}")

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

# Group by language and temperature
grouped = {}
for s in samples:
    lang = s['language']
    temp = s['temperature']
    ref = s.get('reference', '')
    gen = s.get('generated', '')
    if not ref or not gen:
        print(f"  WARNING: skipping sample with empty reference or generated text (lang={lang}, temp={temp})")
        continue
    grouped.setdefault(lang, {}).setdefault(temp, []).append(s)

print(f"Languages found: {list(grouped.keys())}")
for lang in grouped:
    print(f"  {lang} strategies: {list(grouped[lang].keys())} ({sum(len(v) for v in grouped[lang].values())} samples)")

final_metrics = {}
csv_rows = []

for lang, temps in grouped.items():
    final_metrics[lang] = {}
    for temp, items in temps.items():
        bleus, chrfs, rouge_ls, dist1s, dist2s, reps = [], [], [], [], [], []
        for item in items:
            gen = item['generated']
            ref = item['reference']
            bleus.append(sacrebleu.sentence_bleu(gen, [ref]).score)
            chrfs.append(sacrebleu.sentence_chrf(gen, [ref]).score)
            rouge_ls.append(scorer.score(ref, gen)['rougeL'].fmeasure)
            dist1s.append(calc_distinct_n(gen, 1))
            dist2s.append(calc_distinct_n(gen, 2))
            reps.append(calc_repetition_rate(gen))

        mets = {
            "bleu": float(np.mean(bleus)),
            "chrf": float(np.mean(chrfs)),
            "rougeL": float(np.mean(rouge_ls)),
            "repetition_rate": float(np.mean(reps)),
            "distinct_1": float(np.mean(dist1s)),
            "distinct_2": float(np.mean(dist2s))
        }
        final_metrics[lang][temp] = mets
        csv_rows.append({
            'language': lang,
            'strategy': temp,
            **mets
        })
        print(f"  {lang}/{temp}: BLEU={mets['bleu']:.2f}  chrF={mets['chrf']:.2f}  ROUGE-L={mets['rougeL']:.4f}  rep={mets['repetition_rate']:.4f}  D1={mets['distinct_1']:.4f}  D2={mets['distinct_2']:.4f}")

# ---------- write output files ----------

out_dir = OUTPUT_DIR / "completed_generation_metrics"
out_dir.mkdir(parents=True, exist_ok=True)

files_to_write = {
    "hindi_generation_metrics.json": final_metrics.get('hindi', {}),
    "nepali_generation_metrics.json": final_metrics.get('nepali', {}),
}

for fname, data in files_to_write.items():
    fpath = out_dir / fname
    with open(fpath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved: {fpath.resolve()} ({fpath.stat().st_size} bytes)")

csv_path = out_dir / "generation_metrics_comparison.csv"
csv_headers = ['language', 'strategy', 'bleu', 'chrf', 'rougeL', 'repetition_rate', 'distinct_1', 'distinct_2']
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=csv_headers)
    writer.writeheader()
    writer.writerows(csv_rows)
print(f"Saved: {csv_path.resolve()} ({csv_path.stat().st_size} bytes)")

# ---------- summary markdown ----------

summary_lines = [
    "# Generation Metrics Summary\n",
    "## Methodology\n",
    "- **BLEU**: Sentence-level BLEU via `sacrebleu.sentence_bleu`. Internal tokenization (no external tokenizer).\n",
    "- **chrF**: Sentence-level chrF via `sacrebleu.sentence_chrf`. Character n-gram F-score.\n",
    "- **ROUGE-L**: Longest common subsequence F-measure via `rouge_score.RougeScorer` (no stemmer).\n",
    "- **Distinct-1/Distinct-2**: Unique unigrams/bigrams divided by total unigrams/bigrams on whitespace-tokenized text.\n",
    "- **Repetition Rate**: Fraction of tokens in the generated text that appear more than once.\n",
    "- **References**: Gold continuations extracted from the corpus during generation, saved in `generation_samples.json`.\n",
    "- **Tokenization**: BLEU and chrF use sacrebleu internal tokenization. Distinct-N and repetition use whitespace splitting. Both methods handle Hindi and Nepali Unicode identically.\n",
    "\n## Per-Strategy Results\n",
]

higher_better = ['bleu', 'chrf', 'rougeL', 'distinct_1', 'distinct_2']
lower_better = ['repetition_rate']

for lang in ['hindi', 'nepali']:
    if lang not in final_metrics:
        continue
    summary_lines.append(f"### {lang.capitalize()}\n")
    for temp, mets in final_metrics[lang].items():
        summary_lines.append(f"**{temp}**: BLEU={mets['bleu']:.2f}, chrF={mets['chrf']:.2f}, ROUGE-L={mets['rougeL']:.4f}, rep={mets['repetition_rate']:.4f}, D1={mets['distinct_1']:.4f}, D2={mets['distinct_2']:.4f}\n")
    summary_lines.append("\n")

# Best/worst per metric
summary_lines.append("## Best and Worst Strategies\n")
for lang in ['hindi', 'nepali']:
    if lang not in final_metrics:
        continue
    summary_lines.append(f"### {lang.capitalize()}\n")
    temps_data = final_metrics[lang]
    for metric in higher_better:
        best = max(temps_data.items(), key=lambda x: x[1][metric])
        worst = min(temps_data.items(), key=lambda x: x[1][metric])
        summary_lines.append(f"- **{metric}**: best={best[0]} ({best[1][metric]:.4f}), worst={worst[0]} ({worst[1][metric]:.4f})\n")
    for metric in lower_better:
        best = min(temps_data.items(), key=lambda x: x[1][metric])
        worst = max(temps_data.items(), key=lambda x: x[1][metric])
        summary_lines.append(f"- **{metric}** (lower is better): best={best[0]} ({best[1][metric]:.4f}), worst={worst[0]} ({worst[1][metric]:.4f})\n")
    summary_lines.append("\n")

# Hindi vs Nepali comparison
if 'hindi' in final_metrics and 'nepali' in final_metrics:
    summary_lines.append("## Hindi vs Nepali Comparison (Greedy)\n")
    h_greedy = final_metrics['hindi'].get('greedy', {})
    n_greedy = final_metrics['nepali'].get('greedy', {})
    if h_greedy and n_greedy:
        for metric in ['bleu', 'chrf', 'rougeL', 'repetition_rate', 'distinct_1', 'distinct_2']:
            hv = h_greedy.get(metric, 0)
            nv = n_greedy.get(metric, 0)
            summary_lines.append(f"- **{metric}**: Hindi={hv:.4f}, Nepali={nv:.4f}\n")

summary_lines.append("\n## Unavailable Metrics\n")
summary_lines.append("All six metrics (BLEU, chrF, ROUGE-L, repetition_rate, Distinct-1, Distinct-2) were successfully computed. No metrics are unavailable.\n")

summary_path = out_dir / "generation_metrics_summary.md"
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("\n".join(summary_lines))
print(f"Saved: {summary_path.resolve()} ({summary_path.stat().st_size} bytes)")

# ---------- assertions ----------

required_files = [
    out_dir / "hindi_generation_metrics.json",
    out_dir / "nepali_generation_metrics.json",
    out_dir / "generation_metrics_comparison.csv",
    out_dir / "generation_metrics_summary.md",
]
for fp in required_files:
    assert fp.exists(), f"MISSING: {fp}"
    assert fp.stat().st_size > 0, f"EMPTY: {fp}"
    print(f"Verified: {fp.name} exists and is nonempty")

print("\nAll post-hoc generation metrics completed successfully.")
